# FloodNet Data for Nearby Sensors (Sept 29, 2023) - V3

This notebook extends the FloodNet API examples to pull data specifically for the sensors identified in the proximity analysis (`04_image_sensor_proximity.ipynb`) for the major flood event on September 29, 2023.

We will:
 - Load the list of sensors that were near dashcam images
 - Query the FloodNet API for depth data on Sept 29, 2023
 - **Filter out measurement noise (blips, boxes, and gradient spikes) per official documentation**
 - Visualize the flood levels across these locations
 - **Overlay image-based flood annotations as ground truth error bars**
 - Map the sensor locations and their proximity to the captured images

## Setup

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from tqdm.auto import tqdm
import folium
import folium.plugins
import constants as c

plt.rc('figure', figsize=(15, 6))
plt.style.use('ggplot')

## Load Proximity Data & Annotations

We load the metadata for images found within 15m of FloodNet sensors, and the corresponding human-labeled flood levels.

In [ ]:
# Load the metadata created in the proximity analysis
nearby_metadata_path = Path("../../data/revisions/nearby_floodnet/nearby_sensor_images_metadata.csv")

nearby_df = pd.read_csv(nearby_metadata_path)
nearby_sensor_ids = nearby_df['sensor_id'].unique().tolist()

# Load image annotations
annot_path = Path("../../data/revisions/nearby_floodnet/nearby_sensor_images_annotated.csv")


if annot_path.exists():
    df_annot = pd.read_csv(annot_path)
    # Convert timestamps to local time
    df_annot['time'] = pd.to_datetime(df_annot['captured_at'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('America/New_York')
    
    # Map levels to values and error bounds (cm)
    level_map = {
        "Level 0: 0cm": (0.0, 0.0, 0.0),
        "Level 1: 1-15cm": (8.0, 1.0, 15.0),
        "Level 2: 16-40cm": (28.0, 16.0, 40.0),
        "Level 3: 41-60cm": (50.5, 41.0, 60.0),
        "Level 4: > 60cm": (70.0, 60.0, 85.0)
    }
    
    def parse_level(choice):
        mid, low, high = level_map.get(choice, (np.nan, np.nan, np.nan))
        return pd.Series({'annot_mid': mid, 'annot_low': low, 'annot_high': high})
    
    annot_bounds = df_annot['choice'].apply(parse_level)
    # Ensure no duplicate columns if cell is re-run
    cols_to_keep = [c for c in df_annot.columns if c not in annot_bounds.columns]
    df_annot = pd.concat([df_annot[cols_to_keep], annot_bounds], axis=1)
    print(f"Loaded {len(df_annot)} image annotations.")
else:
    df_annot = pd.DataFrame()
    print("Annotations file not found.")

print(f"Loaded {len(nearby_df)} image-sensor pairs across {len(nearby_sensor_ids)} unique sensors.")

In [ ]:
# load floodnet sensors sep29 csv 
sep23_floodnet_sensors = pd.read_csv("../../aggregation/flooding/static/sep23_floodnet_sensor_coordinates.csv")
sep23_floodnet_sensors['deployment_id'].nunique() 

## Noise Filtering Logic

As per the FloodNet documentation, sensors can detect non-flood signals (blips, boxes, complex noise). We implement the official filtering stages with robust handling for sparse data and iterative passes to catch residual noise:

1. **Stage 1 (Low-level noise)**: Depths < 10 mm are assigned to zero.
2. **Stage 2 (Gradient filter)**: Changes > 127 mm/min relative to the last valid point are filtered (more aggressive than the 254 mm/min default to catch street noise).
3. **Stage 3 (Blip/Box filters)**:
   - **Blips**: Momentary jumps where depth returns to baseline quickly ($|D3-D1| / (D2-D1) < 0.25$).
   - **Boxes**: Sudden jumps that return to baseline after a period (pulse detection).
4. **Stage 4 (Outlier Detection)**: Rolling median filter to catch complex noise spikes.

In [ ]:
def apply_floodnet_filters(df):
    '''
    Applies FloodNet heuristic filters to clean measurement noise.
    Enhanced with iterative passes and robust pulse/box detection.
    '''
    if df.empty:
        return df
    
    df = df.sort_values(['deployment_id', 'time']).copy()
    
    # Stage 1: Low-level noise filter
    df['depth_clean_mm'] = df['depth_proc_mm'].where(df['depth_proc_mm'] >= 10.0, 0.0)
    
    def process_sensor(group):
        group = group.sort_index()
        depths = group['depth_clean_mm'].values
        times = group.index
        
        # Stage 2: Iterative Gradient Filter
        # Catch huge jumps even in sparse data by comparing to the LAST VALID point
        if len(depths) > 1:
            last_v = depths[0]
            last_t = times[0]
            for i in range(1, len(depths)):
                if np.isnan(depths[i]): continue
                dt = (times[i] - last_t).total_seconds() / 60.0
                if dt > 0:
                    grad = abs(depths[i] - last_v) / dt
                    if grad > 127.0: # Filter impossible jumps
                        depths[i] = np.nan
                    else:
                        last_v = depths[i]
                        last_t = times[i]
        group['depth_clean_mm'] = depths

        # Stage 3: Heuristic filters (Blip -> Box -> Blip repeats)
        for pass_idx in range(3):
            v_mask = group['depth_clean_mm'].notna()
            v_depths = group.loc[v_mask, 'depth_clean_mm'].values
            v_times = group.index[v_mask]
            if len(v_depths) < 3: break
            
            to_remove = []
            
            # 1. Blip pass
            for i in range(1, len(v_depths) - 1):
                d1, d2, d3 = v_depths[i-1], v_depths[i], v_depths[i+1]
                delta_d = d2 - d1
                if delta_d > 2.0:
                    if abs(d3 - d1) / delta_d < 0.25:
                        to_remove.append(v_times[i])
            
            # 2. Box/Pulse pass
            i = 0
            while i < len(v_depths) - 2:
                d_start = v_depths[i]
                d_jump = v_depths[i+1]
                if d_jump > (d_start + 50.0):
                    j = i + 1
                    box_points = []
                    while j < len(v_depths):
                        if abs(v_depths[j] - d_jump) / d_jump < 0.25:
                            box_points.append(v_times[j])
                            j += 1
                        else:
                            break
                    
                    # Pulse is noise if it returns near baseline or reaches end of window
                    is_noise = False
                    if j < len(v_depths):
                        if v_depths[j] < (d_start + 50.0): # Returns to baseline
                            is_noise = True
                    else:
                        is_noise = True # Reaches end of window
                        
                    if is_noise:
                        to_remove.extend(box_points)
                        i = j
                        continue
                i += 1
            
            group.loc[to_remove, 'depth_clean_mm'] = np.nan
            
        # Stage 4: Outlier Check (Rolling Median)
        group['median'] = group['depth_clean_mm'].rolling(window=5, center=True).median()
        outliers = (group['depth_clean_mm'] - group['median']).abs() > 300.0 # 30cm outlier
        group.loc[outliers, 'depth_clean_mm'] = np.nan
        
        return group

    df = df.groupby('deployment_id', group_keys=False).apply(process_sensor)
    df['depth_cm'] = df['depth_clean_mm'] / 10.0
    return df

## API Helper Functions

Refactored to handle retrieval and proper timezone handling.

In [ ]:
def get_deployments():
    '''Retrieve a table of all sensor deployment locations.'''
    deployments = requests.get("https://api.floodnet.nyc/api/rest/deployments/flood").json()
    if 'error' in deployments:
        raise RuntimeError(str(deployments))

    df = pd.DataFrame(deployments['deployments'])
    df = df.dropna(subset=['location'])
    df['date_deployed'] = pd.to_datetime(df.date_deployed, format='mixed', errors='coerce').dt.tz_localize(tz='America/New_York')
    df['date_down'] = pd.to_datetime(df.date_down, format='mixed', errors='coerce').dt.tz_localize(tz='America/New_York')
    df['coordinates'] = [np.array(x['coordinates'][::-1]) for x in df.location]
    return df

def query_depth_data(deployment_id, start_time, end_time, sensor_name_lookup=None):
    '''Query depth data for a specific sensor.'''
    resp = requests.get(f"https://api.floodnet.nyc/api/rest/deployments/flood/{deployment_id}/depth", params={
        'start_time': start_time,
        'end_time': end_time,
    })
    
    if resp.status_code != 200:
        return pd.DataFrame()

    data = resp.json()
    if 'error' in data or not data.get('depth_data'):
        return pd.DataFrame()

    df_depth = pd.DataFrame(data['depth_data'])
    if 'depth_proc_mm' not in df_depth.columns:
        return pd.DataFrame()
        
    df_depth = df_depth[['deployment_id', 'time', 'depth_proc_mm']]
    
    if sensor_name_lookup is not None:
        df_depth['name'] = df_depth.deployment_id.map(sensor_name_lookup)
    else:
        df_depth['name'] = deployment_id

    # Handle timestamps and timezone
    df_depth['time'] = pd.to_datetime(df_depth['time'], format='ISO8601')
    if df_depth['time'].dt.tz is None:
        df_depth['time'] = df_depth['time'].dt.tz_localize('UTC')
    df_depth['time'] = df_depth['time'].dt.tz_convert('America/New_York')
    
    return df_depth

In [ ]:
# Fetch all deployments and create robust lookups for nearby sensors
df_all_deployments = get_deployments()

# Ensure we include ALL sensors from metadata, even if missing from API list
df_nearby_sensors = pd.DataFrame({"deployment_id": nearby_sensor_ids})
df_nearby_sensors = df_nearby_sensors.merge(df_all_deployments, on="deployment_id", how="left")

# Fallback for missing names: use the ID
df_nearby_sensors["name"] = df_nearby_sensors["name"].fillna(df_nearby_sensors["deployment_id"])

# Create a name lookup (deployment_id -> name)
sensor_name_lookup = df_nearby_sensors.set_index("deployment_id")["name"].to_dict()

print(f"Found metadata for {len(df_nearby_sensors)} nearby sensors. ({df_nearby_sensors["name"].isna().sum()} missing API info)")


## Bulk Data Retrieval: September 29, 2023

In [ ]:
# Fetch data starting from midnight local time on the 29th
start_time = "2023-09-29T00:00:00-04:00"
end_time = "2023-09-30T00:00:00-04:00"

raw_data_list = []
for dep_id in tqdm(nearby_sensor_ids, desc="Querying sensors"):
    df_dep = query_depth_data(dep_id, start_time, end_time, sensor_name_lookup)
    if not df_dep.empty:
        raw_data_list.append(df_dep)

if raw_data_list:
    df_all_raw = pd.concat(raw_data_list).set_index('time')
    
    # Apply Filters
    print("Applying noise filters...")
    df_depth = apply_floodnet_filters(df_all_raw)
    
    
    print(f"Retrieved and cleaned {len(df_depth)} data points across {len(df_depth.deployment_id.unique())} sensors.")
else:
    df_depth = pd.DataFrame()
    print("No data found.")

In [ ]:
df_depth['deployment_id'].nunique()

## Exporting Data

In [ ]:
if not df_depth.empty:
    output_csv = Path("../../data/revisions/nearby_floodnet/sept29_nearby_sensors_depth_data_cleaned.csv")
    df_depth.to_csv(output_csv)
    print(f"Exported cleaned depth data to {output_csv}")

In [ ]:
from transformers import pipeline
from functools import lru_cache
import torch

INFERENCE=False
if INFERENCE:

    # Initialize NLI pipeline for textual entailment-based flood level extraction
    # Using facebook/bart-large-mnli which is effective for zero-shot classification
    print("Loading NLI model for flood level extraction...")
    nli_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)

    # Define flood level hypotheses for NLI classification
    FLOOD_LEVEL_HYPOTHESES = {
        "Level 0: 0cm": [
            "There is no flooding visible.",
            "The water level is zero.",
            "No floodwater is present.",
            "The area is dry with no standing water.",
        ],
        "Level 1: 1-15cm": [
            "There is minor flooding with shallow water.",
            "The water level is ankle-deep or less.",
            "There is slight flooding between 1 and 15 centimeters.",
            "Shallow puddles or minor flooding is visible.",
        ],
        "Level 2: 16-40cm": [
            "There is moderate flooding with water reaching mid-calf level.",
            "The water level is between 16 and 40 centimeters.",
            "Moderate flooding is present, roughly knee-deep.",
            "Significant standing water covering the ground.",
        ],
        "Level 3: 41-60cm": [
            "There is significant flooding with water reaching knee to thigh level.",
            "The water level is between 41 and 60 centimeters.",
            "Deep flooding is present, roughly waist-deep.",
            "Severe flooding with water above knee height.",
        ],
        "Level 4: > 60cm": [
            "There is severe flooding with water above 60 centimeters.",
            "The water level is above waist height.",
            "Extreme flooding with deep water levels.",
            "Very deep flooding exceeding 60 centimeters.",
        ],
    }

    # Create flattened list of all hypotheses with their corresponding levels
    ALL_HYPOTHESES = []
    HYPOTHESIS_TO_LEVEL = {}
    for level, hypotheses in FLOOD_LEVEL_HYPOTHESES.items():
        for h in hypotheses:
            ALL_HYPOTHESES.append(h)
            HYPOTHESIS_TO_LEVEL[h] = level


    def parse_vlm_level_nli(model_response: str, classifier=None) -> dict:
        """
        Parse flood level from VLM model response using NLI textual entailment.
        
        Uses zero-shot classification to determine which flood level hypothesis
        the model response most strongly entails.
        
        Args:
            model_response: The text response from the VLM model
            classifier: The NLI pipeline (uses global nli_classifier if None)
        
        Returns:
            dict with keys: parsed_choice, annot_mid, annot_low, annot_high, nli_scores
        """
        if classifier is None:
            classifier = nli_classifier
        
        response_text = str(model_response).strip()
        
        # Handle empty or very short responses
        if len(response_text) < 10:
            return {
                "parsed_choice": "N/A",
                "annot_mid": np.nan,
                "annot_low": np.nan,
                "annot_high": np.nan,
                "nli_scores": {},
            }
        
        # Use zero-shot classification with all hypotheses
        # This computes entailment scores for each hypothesis
        result = classifier(
            response_text,
            candidate_labels=ALL_HYPOTHESES,
            multi_label=True,  # Allow independent scores for each hypothesis
        )
        
        # Aggregate scores by flood level (take max score per level)
        level_scores = {}
        for label, score in zip(result["labels"], result["scores"]):
            level = HYPOTHESIS_TO_LEVEL[label]
            if level not in level_scores or score > level_scores[level]:
                level_scores[level] = score
        
        # Select the level with highest score
        if level_scores:
            best_level = max(level_scores, key=level_scores.get)
            best_score = level_scores[best_level]
            
            # Apply confidence threshold - if best score is too low, mark as inconclusive
            if best_score < 0.3:
                return {
                    "parsed_choice": "N/A",
                    "annot_mid": np.nan,
                    "annot_low": np.nan,
                    "annot_high": np.nan,
                    "nli_scores": level_scores,
                }
            
            # Map to depth values
            mid, low, high = level_map.get(best_level, (np.nan, np.nan, np.nan))
            return {
                "parsed_choice": best_level,
                "annot_mid": mid,
                "annot_low": low,
                "annot_high": high,
                "nli_scores": level_scores,
            }
        
        return {
            "parsed_choice": "N/A",
            "annot_mid": np.nan,
            "annot_low": np.nan,
            "annot_high": np.nan,
            "nli_scores": {},
        }


    def apply_nli_parsing(df: pd.DataFrame, response_col: str = "model_response") -> pd.DataFrame:
        """
        Apply NLI-based flood level parsing to a DataFrame.
        
        Args:
            df: DataFrame with model responses
            response_col: Column name containing model responses
        
        Returns:
            DataFrame with added columns: parsed_choice, annot_mid, annot_low, annot_high
        """
        results = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="NLI parsing"):
            parsed = parse_vlm_level_nli(row[response_col])
            results.append(parsed)
        
        result_df = pd.DataFrame(results)
        
        # Add results to original dataframe
        for col in ["parsed_choice", "annot_mid", "annot_low", "annot_high"]:
            df[col] = result_df[col].values
        
        return df


    print("NLI flood level extractor ready.")

In [ ]:
# Load VLM Basic Predictions 
if INFERENCE:
    vlm_basic_path = Path("../../data/revisions/nearby_floodnet/bayflood_nearby_floodnet_relative_basic_20260111_110205.parquet")
    if vlm_basic_path.exists():
        df_vlm_basic = pd.read_parquet(vlm_basic_path)

        # Merge with proximity metadata to get sensor IDs and timestamps
        # We drop existing image_path to use the metadata version
        meta_cols = ["image_path", "sensor_id", "captured_at", "frame_id"]
        df_vlm_basic = df_vlm_basic.drop(columns=["image_path"], errors="ignore")
        df_vlm_basic = df_vlm_basic.merge(nearby_df[meta_cols], left_on="sample_id", right_on="frame_id")
        
        df_vlm_basic["time"] = pd.to_datetime(df_vlm_basic["captured_at"], unit="ms").dt.tz_localize("UTC").dt.tz_convert("America/New_York")
        
        # Apply NLI-based flood level extraction (replaces brittle regex heuristics)
        df_vlm_basic = apply_nli_parsing(df_vlm_basic, response_col="model_response")

        # drop rows from sensors that aren't present in df_depth 
        df_vlm_basic = df_vlm_basic[df_vlm_basic['sensor_id'].isin(df_depth['deployment_id'])]
        print(f"Loaded {len(df_vlm_basic)} VLM basic predictions. ({df_vlm_basic['annot_mid'].isna().sum()} inconclusive)")
else: 
    df_vlm_basic = pd.read_csv("../../data/revisions/nearby_floodnet/bayflood_rde_basic_annotated.csv")
        

    

In [ ]:
# Load VLM Advanced predictions
if INFERENCE:
    vlm_adv_path = Path("../../data/revisions/nearby_floodnet/bayflood_nearby_floodnet_relative_advanced_20260111_110446.parquet")
    if vlm_adv_path.exists():
        df_vlm_adv = pd.read_parquet(vlm_adv_path)
        
        # Merge with proximity metadata to get sensor IDs and timestamps
        # We drop existing image_path to use the metadata version
        meta_cols = ["image_path", "sensor_id", "captured_at", "frame_id"]
        df_vlm_adv = df_vlm_adv.drop(columns=["image_path"], errors="ignore")
        df_vlm_adv = df_vlm_adv.merge(nearby_df[meta_cols], left_on="sample_id", right_on="frame_id")
        
        df_vlm_adv["time"] = pd.to_datetime(df_vlm_adv["captured_at"], unit="ms").dt.tz_localize("UTC").dt.tz_convert("America/New_York")
        
        # Apply NLI-based flood level extraction (replaces brittle regex heuristics)
        df_vlm_adv = apply_nli_parsing(df_vlm_adv, response_col="model_response")
        
        # drop rows from sensors that aren't present in df_depth 
        df_vlm_adv = df_vlm_adv[df_vlm_adv['sensor_id'].isin(df_depth['deployment_id'])]
        print(f"Loaded {len(df_vlm_adv)} VLM advanced predictions. ({df_vlm_adv['annot_mid'].isna().sum()} inconclusive)")
else:
    df_vlm_adv = pd.read_csv("../../data/revisions/nearby_floodnet/bayflood_rde_advanced_annotated.csv")

In [ ]:
# how many unique sensors have non-inconclusive VLM predictions? (advanced)
df_vlm_adv[df_vlm_adv['annot_mid'].notna()]['sensor_id'].nunique()


In [ ]:
# NLI is unreliable, use annotations in 'sentiment' column from Label Studio 
level_dict = {
    "Level 0": {"pred_low": 0, "pred_mid": 0, "pred_high": 0},
    "Level 1": {"pred_low": 1, "pred_mid": 8, "pred_high": 15},
    "Level 2": {"pred_low": 16, "pred_mid": 28, "pred_high": 40},
    "Level 3": {"pred_low": 41, "pred_mid": 50.5, "pred_high": 60},
    "Level 4": {"pred_low": 60, "pred_mid": 70, "pred_high": 85},
}

df_vlm_adv['pred_low'] = df_vlm_adv['sentiment'].map(lambda x: level_dict.get(x, {}).get('pred_low', np.nan))
df_vlm_adv['pred_mid'] = df_vlm_adv['sentiment'].map(lambda x: level_dict.get(x, {}).get('pred_mid', np.nan))
df_vlm_adv['pred_high'] = df_vlm_adv['sentiment'].map(lambda x: level_dict.get(x, {}).get('pred_high', np.nan))

df_vlm_basic['pred_low'] = df_vlm_basic['sentiment'].map(lambda x: level_dict.get(x, {}).get('pred_low', np.nan))
df_vlm_basic['pred_mid'] = df_vlm_basic['sentiment'].map(lambda x: level_dict.get(x, {}).get('pred_mid', np.nan))
df_vlm_basic['pred_high'] = df_vlm_basic['sentiment'].map(lambda x: level_dict.get(x, {}).get('pred_high', np.nan))



In [ ]:
df_vlm_basic.info()

In [ ]:
# drop annot_low, annot_mid, annot_high
df_vlm_adv = df_vlm_adv.drop(columns=['annot_low', 'annot_mid', 'annot_high'])
df_vlm_basic = df_vlm_basic.drop(columns=['annot_low', 'annot_mid', 'annot_high'])


# remerge clean annotations into df_vlm_adv and df_vlm_basic
df_vlm_adv = df_vlm_adv.merge(df_annot[['frame_id','annot_mid', 'annot_low', 'annot_high', 'choice']], on='frame_id')
df_vlm_basic = df_vlm_basic.merge(df_annot[['frame_id', 'annot_mid', 'annot_low', 'annot_high', 'choice']], on='frame_id')





In [ ]:
df_vlm_adv.info()

In [ ]:
df_vlm_adv.columns

In [ ]:
df_vlm_basic['annot_mid'].isna().sum()

In [ ]:
df_vlm_adv['annot_mid'].isna().sum()

In [ ]:
df_vlm_basic_by_gt_level = df_vlm_basic.groupby('choice')
# print value counts of sentiment for each annot_mid level 
for level, group in df_vlm_basic_by_gt_level:
    print(f"{level}:")
    print(group['sentiment'].value_counts())
    print("\n")



In [ ]:
df_vlm_adv_by_gt_level = df_vlm_adv.groupby('choice')
# print value counts of sentiment for each annot_mid level 
for level, group in df_vlm_adv_by_gt_level:
    print(f"{level}:")
    print(group['sentiment'].value_counts())
    print("\n")




In [ ]:
# compute agreement rate between VLM and human annotations
def compute_agreement_rate(df_vlm):
    # compute agreement rate between VLM and human annotations -- 'sentiment' vs 'choice' (split choice at :)
    df_vlm['choice_split'] = df_vlm['choice'].str.split(':').str[0]
    # just report the agreement rate between 'sentiment' and 'choice_split' -- ie the number of rows where sentiment == choice_split / total number of rows
    print(f"Agreement rate between 'sentiment' and 'choice_split': {(df_vlm['sentiment'] == df_vlm['choice_split']).mean()}")
    return (df_vlm['sentiment'] == df_vlm['choice_split']).mean()

compute_agreement_rate(df_vlm_adv)
compute_agreement_rate(df_vlm_basic)    



In [ ]:
df_vlm_basic['sentiment'].value_counts()

In [ ]:
df_vlm_adv['sentiment'].value_counts()

In [ ]:
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.dates as mdates
# import path_effects
import matplotlib.patheffects as path_effects


# Identify sensors with any prediction/annotation > Level 0
prominent_sensors = set()
for df_source in [df_annot, df_vlm_adv]:
    if not df_source.empty:
        merged = df_source.merge(df_nearby_sensors[["deployment_id", "name"]], left_on="sensor_id", right_on="deployment_id")
        flooded = merged[merged.annot_mid > 0]["name"].unique()
        prominent_sensors.update(flooded)


In [ ]:
prominent_sensors

In [ ]:
import matplotlib.patheffects as path_effects
import string
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch
import matplotlib.gridspec as gridspec

image_crop_config = {
}

# Helper to handle >26 panels (A, B, ..., Z, AA, AB...)
def get_panel_label(idx):
    alphabet = string.ascii_uppercase
    if idx < 26:
        return alphabet[idx]
    else:
        return alphabet[idx // 26 - 1] + alphabet[idx % 26]

# --- FINAL NATURE-LEVEL REFINED SMALL MULTIPLES ---

excluded_sensors = [] 

# Identify the top 8 sensors with highest flooding levels
sensor_peaks = {}

# 1. Peaks from FloodNet API
if not df_depth.empty:
    api_peaks = df_depth.groupby("name")["depth_cm"].max().to_dict()
    for s, val in api_peaks.items():
        sensor_peaks[s] = max(sensor_peaks.get(s, 0), val)

# 2. Peaks from Human Annotations
if not df_annot.empty:
    annot_merged = df_annot.merge(df_nearby_sensors[["deployment_id", "name"]], left_on="sensor_id", right_on="deployment_id")
    annot_peaks = annot_merged.groupby("name")["annot_mid"].max().to_dict()
    for s, val in annot_peaks.items():
        sensor_peaks[s] = max(sensor_peaks.get(s, 0), val)

# 3. Peaks from VLM Predictions
if not df_vlm_adv.empty:
    vlm_merged = df_vlm_adv.merge(df_nearby_sensors[["deployment_id", "name"]], left_on="sensor_id", right_on="deployment_id")
    vlm_peaks = vlm_merged.groupby("name")["annot_mid"].max().to_dict()
    for s, val in vlm_peaks.items():
        sensor_peaks[s] = max(sensor_peaks.get(s, 0), val)

# Select top 8 unique names
sorted_sensors = sorted([s for s in sensor_peaks if isinstance(s, str) and s not in excluded_sensors], 
                        key=lambda x: sensor_peaks[x], reverse=True)
# Filter active_sensors to only include sensors that were deployed and active on Sept 29, 2023
event_date = pd.Timestamp("2023-09-29").tz_localize("America/New_York")
active_on_date = []
for sname in sorted_sensors:
    # Get deployment info for this sensor name
    dep_info = df_nearby_sensors[df_nearby_sensors.name == sname]
    if not dep_info.empty:
        # Check if any deployment for this name was active on the event date
        is_active = False
        for _, row in dep_info.iterrows():
            d_start = row.get('date_deployed')
            d_end = row.get('date_down')
            if pd.notna(d_start) and d_start <= event_date:
                if pd.isna(d_end) or d_end >= event_date:
                    is_active = True
                    break
        if is_active:
            active_on_date.append(sname)

active_sensors = active_on_date[:12]
active_sensors = sorted(active_sensors) # Alphabetical sequence for A-L # Alphabetical sequence for A-H

if active_sensors:
    plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
    plt.rcParams["font.family"] = "sans-serif"
    plt.style.use("default")
    
    num_panels = len(active_sensors)
    num_rows_plot = num_panels
    num_rows_img = (num_panels + 1) // 2
    
    fig = plt.figure(figsize=(16, 1.2 * num_rows_plot), dpi=300)
    gs_main = gridspec.GridSpec(num_rows_plot, 2, width_ratios=[1.8, 2.0], hspace=0.1, wspace=0.15)
    gs_imgs = gridspec.GridSpecFromSubplotSpec(num_rows_img, 2, subplot_spec=gs_main[:, 1], hspace=0, wspace=0)
    
    cmap = cm.get_cmap("tab20")
    plot_axes = {} 
    img_axes = {}  
    
    for i, sname in enumerate(active_sensors):
        ax_plot = fig.add_subplot(gs_main[i, 0])
        plot_axes[i] = ax_plot
        color = cmap(i % 20)
        
        # 1. Plot Sensor Data
        if not df_depth.empty and sname in df_depth["name"].values:
            s_data = df_depth[df_depth.name == sname].sort_index()
            # 1. Round sensor timestamps to nearest minute to align with the 1-min grid
            s_data_rounded = s_data.copy()
            s_data_rounded.index = s_data_rounded.index.round('1min').tz_localize(None)
            # Remove duplicates created by rounding if any
            s_data_rounded = s_data_rounded[~s_data_rounded.index.duplicated(keep='first')]
            
            # 2. Create the full 24h range and reindex
            full_range = pd.date_range(start="2023-09-29 00:00:00", end="2023-09-29 23:59:00", freq='1min')
            s_data_full = s_data_rounded.reindex(full_range)
            
            # 3. Interpolate internal gaps, then fill leading/trailing gaps with 0.0
            interp_depths = s_data_full.depth_cm.interpolate(method='linear').fillna(0.0)
            
            # 4. Plot valid data segments (using original high-res timestamps for accuracy)
            naive_index = s_data.index.tz_localize(None)
            ax_plot.plot(naive_index, s_data.depth_cm, color=color, linewidth=2.0, alpha=0.9, zorder=10)
            
            # 5. Plot background layer (dashed) using the aligned 1-min grid
            ax_plot.plot(full_range, interp_depths, color=color, linewidth=1.0, alpha=0.3, linestyle='--', zorder=9)
        
        dep_ids = df_nearby_sensors[df_nearby_sensors.name == sname].deployment_id.unique()
        
        # 2. Plot VLM Predictions (Squares)
        if not df_vlm_adv.empty:
            s_vlm = df_vlm_adv[df_vlm_adv.sensor_id.isin(dep_ids)].copy()
            # Debug: print count
            # print(f"{sname}: Found {len(s_vlm)} VLM rows")
            s_vlm["annot_mid"] = pd.to_numeric(s_vlm["annot_mid"], errors="coerce")
            if not s_vlm.empty:
                m_times = pd.to_datetime(s_vlm['time'])
                eb_vlm = ax_plot.errorbar(m_times, s_vlm["annot_mid"], 
                                 yerr=[s_vlm["annot_mid"]-s_vlm["annot_low"], s_vlm["annot_high"]-s_vlm["annot_mid"]],
                                 fmt="s", color=color, markersize=8, capsize=2, elinewidth=1.0,
                                 markerfacecolor="none", markeredgecolor=color, markeredgewidth=1.2, zorder=20)
                eb_vlm.lines[0].set_path_effects([path_effects.withStroke(linewidth=2.5, foreground="black")])

        # 3. Plot Ground Truth (Stars)
        if not df_annot.empty:
            s_annot = df_annot[df_annot.sensor_id.isin(dep_ids)].copy()
            # Debug: print count
            # print(f"{sname}: Found {len(s_annot)} GT rows")
            s_annot["annot_mid"] = pd.to_numeric(s_annot["annot_mid"], errors="coerce")
            if not s_annot.empty:
                m_times = s_annot["time"].dt.tz_localize(None)
                ax_plot.errorbar(m_times, s_annot["annot_mid"], 
                                 yerr=[s_annot["annot_mid"]-s_annot["annot_low"], s_annot["annot_high"]-s_annot["annot_mid"]],
                                 fmt="*", color=color, markersize=10, capsize=2, elinewidth=1.0,
                                 markeredgecolor="black", markeredgewidth=0.4, zorder=30)

        label = get_panel_label(i)
        ax_plot.text(-0.1, 0.98, label, transform=ax_plot.transAxes, fontsize=14, fontweight="bold", va="top", ha="right")
        ax_plot.text(0.02, 0.94, sname, transform=ax_plot.transAxes, fontsize=10, fontweight="bold", color="#333333", va="top")
        
        ax_plot.set_xlim(pd.Timestamp("2023-09-29 00:00:00"), pd.Timestamp("2023-09-29 22:00:00"))
        ax_plot.set_ylim(-5, 105)
        ax_plot.set_yticks([0, 20, 40, 60, 80, 100])
        ax_plot.set_ylabel("water level [cm]", fontsize=7, labelpad=1)
        ax_plot.tick_params(axis="y", labelsize=6)
        ax_plot.grid(True, axis="x", linestyle="--", alpha=0.3, color="#bbbbbb", zorder=0)
        ax_plot.spines["top"].set_visible(False)
        ax_plot.spines["right"].set_visible(False)
        
        if i < num_rows_plot - 1: ax_plot.tick_params(labelbottom=False, bottom=True)
        else:
            ax_plot.xaxis.set_major_locator(mdates.HourLocator(interval=4))
            ax_plot.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
            ax_plot.set_xlabel("Time (Local)", fontsize=12)
            ax_plot.tick_params(axis="x", labelsize=10)

    # 2. Create Image Grid
    for i in range(num_panels):
        r_img, c_img = i // 2, i % 2
        ax_img = fig.add_subplot(gs_imgs[r_img, c_img])
        sname = active_sensors[i]
        dep_ids = df_nearby_sensors[df_nearby_sensors.name == sname].deployment_id.unique()
        
        m_list = []
        if not df_annot.empty:
            sa = df_annot[df_annot.sensor_id.isin(dep_ids)].copy()
            if not sa.empty: m_list.append(sa.rename(columns={"image":"p"}).assign(src="GT")[["annot_mid","p","time","src"]])
        if not df_vlm_adv.empty:
            sv = df_vlm_adv[df_vlm_adv.sensor_id.isin(dep_ids)].copy()
            if not sv.empty: m_list.append(sv.rename(columns={"image_path":"p"}).assign(src="VLM")[["annot_mid","p","time","src"]])
        
        if m_list:
            df_m = pd.concat(m_list)
            best = df_m.sort_values(["annot_mid", "time"], ascending=[False, True]).iloc[0]
            orig_path = str(best["p"]).split("?d=")[1] if "?d=" in str(best["p"]) else str(best["p"])
            frame_id = Path(orig_path).stem
            nlbx_path = f"/share/ju/nexar_data/training_datasets/street_flooding/all_no_letterboxing/nlbx_{frame_id}.jpg"
            final_img_path = nlbx_path if Path(nlbx_path).exists() else orig_path
            
            if Path(final_img_path).exists():
                img = plt.imread(final_img_path)
                h, w = img.shape[:2]
                y1f, y2f, x1f, x2f = image_crop_config.get(sname, [0.0, 1.0, 0.0, 1.0])
                img_crop = img[int(y1f*h):int(y2f*h), int(x1f*w):int(x2f*w)]
                ax_img.imshow(img_crop, aspect="auto", interpolation="lanczos")
                ax_img.text(0.05, 0.95, get_panel_label(i), transform=ax_img.transAxes, fontsize=12, fontweight="bold", color="white", va="top", ha="left", path_effects=[path_effects.withStroke(linewidth=2, foreground="black")])
                ax_img.text(0.95, 0.05, f"{best["time"].strftime("%H:%M")} Peak", transform=ax_img.transAxes, color="white", fontsize=10, fontweight="bold", ha="right", va="bottom", path_effects=[path_effects.withStroke(linewidth=2, foreground="black")])
                plot_axes[i].axvline(best["time"].tz_localize(None), color=cmap(i % 20), linestyle=":", alpha=0.6, linewidth=1, zorder=5)
        
        for spine in ax_img.spines.values():
            spine.set_edgecolor(cmap(i % 20))
            spine.set_linewidth(0.5)
        ax_img.set_xticks([]); ax_img.set_yticks([])

    legend_elements = [
        Line2D([0], [0], color="#666666", lw=2.0, label="FloodNet Sensor API"),
        Line2D([0], [0], marker="*", color="#666666", label="Human Ground Truth", markerfacecolor="#666666", markersize=10, linestyle="None"),
        Line2D([0], [0], marker="s", color="#666666", label="VLM Prediction (Adv)", markerfacecolor="none", markeredgecolor="#666666", markeredgewidth=1.2, markersize=9, linestyle="None")
    ]
    legend_elements[2].set_path_effects([path_effects.withStroke(linewidth=2, foreground="black")])
    fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 0.94), ncol=3, frameon=False, fontsize=12)
    
    plt.savefig(f"{c.PAPER_PATH}/figures/nature_small_multiples_final.pdf", bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
# --- ALTERNATE JOINT PLOT (Two Columns of Curves, No Images) ---
from matplotlib.patches import Patch

# Helper to handle >26 panels (A, B, ..., Z, AA, AB...)
def get_panel_label(idx):
    alphabet = string.ascii_uppercase
    if idx < 26:
        return alphabet[idx]
    else:
        return alphabet[idx // 26 - 1] + alphabet[idx % 26]

excluded_sensors = []

# Identify the top sensors with highest flooding levels
sensor_peaks = {}

# 1. Peaks from FloodNet API
if not df_depth.empty:
    api_peaks = df_depth.groupby("name")["depth_cm"].max().to_dict()
    for s, val in api_peaks.items():
        sensor_peaks[s] = max(sensor_peaks.get(s, 0), val)

# 2. Peaks from Human Annotations
if not df_annot.empty:
    annot_merged = df_annot.merge(df_nearby_sensors[["deployment_id", "name"]], left_on="sensor_id", right_on="deployment_id")
    annot_peaks = annot_merged.groupby("name")["annot_mid"].max().to_dict()
    for s, val in annot_peaks.items():
        sensor_peaks[s] = max(sensor_peaks.get(s, 0), val)

# 3. Peaks from VLM Predictions
if not df_vlm_adv.empty:
    vlm_merged = df_vlm_adv.merge(df_nearby_sensors[["deployment_id", "name"]], left_on="sensor_id", right_on="deployment_id")
    vlm_peaks = vlm_merged.groupby("name")["annot_mid"].max().to_dict()
    for s, val in vlm_peaks.items():
        sensor_peaks[s] = max(sensor_peaks.get(s, 0), val)

# Select top unique names sorted by peak
sorted_sensors = sorted([s for s in sensor_peaks if isinstance(s, str) and s not in excluded_sensors], 
                        key=lambda x: sensor_peaks[x], reverse=True)

# Filter to sensors that were deployed and active on Sept 29, 2023
event_date = pd.Timestamp("2023-09-29").tz_localize("America/New_York")
active_on_date = []
for sname in sorted_sensors:
    dep_info = df_nearby_sensors[df_nearby_sensors.name == sname]
    if not dep_info.empty:
        is_active = False
        for _, row in dep_info.iterrows():
            d_start = row.get('date_deployed')
            d_end = row.get('date_down')
            if pd.notna(d_start) and d_start <= event_date:
                if pd.isna(d_end) or d_end >= event_date:
                    is_active = True
                    break
        if is_active:
            active_on_date.append(sname)

active_sensors = active_on_date[:12]
active_sensors = sorted(active_sensors)  # Alphabetical sequence for A-L

if active_sensors:
    plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
    plt.rcParams["font.family"] = "sans-serif"
    plt.style.use("default")
    
    num_panels = len(active_sensors)
    num_cols = 2
    num_rows = (num_panels + 1) // num_cols
    
    # Use GridSpec for precise control: [y-axis label column, left plot, right plot]
    fig = plt.figure(figsize=(16, 2.2 * num_rows), dpi=300)
    gs = fig.add_gridspec(num_rows, 3, width_ratios=[0.1, 1, 1], wspace=0.015, hspace=0.08)
    
    cmap = cm.get_cmap("tab20")
    row_axes = {}  # Store axes by row for shared y-axis
    
    for i, sname in enumerate(active_sensors):
        row_idx = i // num_cols
        col_idx = i % num_cols
        
        # Create subplot - share y with left plot of same row if this is right column
        if col_idx == 0:
            ax_plot = fig.add_subplot(gs[row_idx, 1])
            row_axes[row_idx] = ax_plot
        else:
            ax_plot = fig.add_subplot(gs[row_idx, 2], sharey=row_axes[row_idx])
        
        color = cmap(i % 20)
        
        # 1. Plot Sensor Data
        if not df_depth.empty and sname in df_depth["name"].values:
            s_data = df_depth[df_depth.name == sname].sort_index()
            s_data_rounded = s_data.copy()
            s_data_rounded.index = s_data_rounded.index.round('1min').tz_localize(None)
            s_data_rounded = s_data_rounded[~s_data_rounded.index.duplicated(keep='first')]
            
            full_range = pd.date_range(start="2023-09-29 00:00:00", end="2023-09-29 23:59:00", freq='1min')
            s_data_full = s_data_rounded.reindex(full_range)
            interp_depths = s_data_full.depth_cm.interpolate(method='linear').fillna(0.0)
            
            naive_index = s_data.index.tz_localize(None)
            ax_plot.plot(naive_index, s_data.depth_cm, color=color, linewidth=2.0, alpha=0.9, zorder=10)
            ax_plot.plot(full_range, interp_depths, color=color, linewidth=1.0, alpha=0.3, linestyle='--', zorder=9)
        
        dep_ids = df_nearby_sensors[df_nearby_sensors.name == sname].deployment_id.unique()
        
        # 2. Plot VLM Predictions (Squares)
        if not df_vlm_adv.empty:
            s_vlm = df_vlm_adv[df_vlm_adv.sensor_id.isin(dep_ids)].copy()
            s_vlm["annot_mid"] = pd.to_numeric(s_vlm["annot_mid"], errors="coerce")
            if not s_vlm.empty:
                m_times = pd.to_datetime(s_vlm['time'])
                eb_vlm = ax_plot.errorbar(m_times, s_vlm["annot_mid"], 
                                 yerr=[s_vlm["annot_mid"]-s_vlm["annot_low"], s_vlm["annot_high"]-s_vlm["annot_mid"]],
                                 fmt="s", color=color, markersize=8, capsize=2, elinewidth=1.0,
                                 markerfacecolor="none", markeredgecolor=color, markeredgewidth=1.2, zorder=20)
                eb_vlm.lines[0].set_path_effects([path_effects.withStroke(linewidth=2.5, foreground="black")])

        # 3. Plot Ground Truth (Stars)
        if not df_annot.empty:
            s_annot = df_annot[df_annot.sensor_id.isin(dep_ids)].copy()
            s_annot["annot_mid"] = pd.to_numeric(s_annot["annot_mid"], errors="coerce")
            if not s_annot.empty:
                m_times = pd.to_datetime(s_annot['time'])
                ax_plot.errorbar(m_times, s_annot["annot_mid"], 
                                 yerr=[s_annot["annot_mid"]-s_annot["annot_low"], s_annot["annot_high"]-s_annot["annot_mid"]],
                                 fmt="*", color=color, markersize=10, capsize=2, elinewidth=1.0,
                                 markeredgecolor="black", markeredgewidth=0.4, zorder=30)
                # Add grey hatched region for GT stars without matching VLM prediction
                if not df_vlm_adv.empty:
                    s_vlm = df_vlm_adv[df_vlm_adv.sensor_id.isin(dep_ids)].copy()
                    vlm_times = pd.to_datetime(s_vlm[s_vlm["annot_mid"].notna()]["time"])
                    for gt_time in m_times:
                        has_vlm = any(abs((gt_time - vt).total_seconds()) < 1.0 for vt in vlm_times)
                        if not has_vlm:
                            ax_plot.axvspan(gt_time - pd.Timedelta(minutes=10), gt_time + pd.Timedelta(minutes=10), 
                                            facecolor="grey", alpha=0.2, hatch="//", edgecolor="grey", linewidth=0, zorder=5)

        # Panel label and sensor name
        label = get_panel_label(i)
        ax_plot.text(0.02, 0.94, f"({label})  {sname}", transform=ax_plot.transAxes, fontsize=12, fontweight="bold", color="#333333", va="top")
        
        ax_plot.set_xlim(pd.Timestamp("2023-09-29 00:00:00"), pd.Timestamp("2023-09-29 22:00:00"))
        ax_plot.set_ylim(-5, 105)
        
        # Grid lines extend across the full width
        ax_plot.grid(True, axis="y", linestyle="--", alpha=0.3, color="#bbbbbb", zorder=0)
        ax_plot.grid(True, axis="x", linestyle="--", alpha=0.3, color="#bbbbbb", zorder=0)
        ax_plot.spines["top"].set_visible(False)
        
        # Handle y-axis: only show ticks on left column, hide on right
        if col_idx == 0:
            ax_plot.set_yticks([0, 20, 40, 60, 80, 100])
            ax_plot.tick_params(axis="y", labelsize=12)
            ax_plot.spines["right"].set_visible(False)
        else:
            ax_plot.tick_params(axis="y", labelleft=False, left=False)
            ax_plot.spines["left"].set_visible(False)
            ax_plot.spines["right"].set_visible(False)
        
        # X-axis: only show labels on bottom row
        if row_idx == num_rows - 1:
            ax_plot.xaxis.set_major_locator(mdates.HourLocator(interval=4))
            ax_plot.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
            ax_plot.set_xlabel("Time (Local)", fontsize=14)
            ax_plot.tick_params(axis="x", labelsize=12)
        else:
            ax_plot.tick_params(labelbottom=False)
    
    # Add shared y-axis label for each row (in the narrow left column)
    for row_idx in range(num_rows):
        ax_ylabel = fig.add_subplot(gs[row_idx, 0])
        ax_ylabel.set_axis_off()
        #ax_ylabel.text(0, 0.5, "water level [cm]", transform=ax_ylabel.transAxes, 
                       #fontsize=14, rotation=90, va="center", ha="center")

    # only add y-axis label once in the vertical middle of the plot, 
    fig.text(0.12, 0.5, "Water Level [cm]", transform=fig.transFigure, 
             fontsize=14, rotation=90, va="center", ha="center")

    legend_elements = [
        Line2D([0], [0], color="#666666", lw=2.0, label="FloodNet Sensor API"),
        Line2D([0], [0], marker="*", color="#666666", label="Human Ground Truth", markerfacecolor="#666666", markersize=14, linestyle="None"),
        Line2D([0], [0], marker="s", color="#666666", label="VLM Prediction (Adv.)", markerfacecolor="none", markeredgecolor="#666666", markeredgewidth=1.2, markersize=14, linestyle="None"),
        Patch(lw=2.0, label="Inconclusive VLM Predictions", hatch="//", facecolor="none", edgecolor="grey", hatch_linewidth=2)
    ]
    legend_elements[2].set_path_effects([path_effects.withStroke(linewidth=2, foreground="black")])
    fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=4, frameon=False, fontsize=14)

   
    
    plt.savefig(f"{c.PAPER_PATH}/figures/nature_small_multiples_curves_only.png", bbox_inches="tight", dpi=300)
    plt.show()

## Summary of Peak Flood Events and VLM Performance

The following table summarizes the peak water levels recorded by sensors, human annotations, and VLM predictions for the key locations during the September 29, 2023 storm.

In [ ]:
import pandas as pd
from IPython.display import display

stats = []
for sname in active_sensors:
    dep_ids = df_nearby_sensors[df_nearby_sensors.name == sname].deployment_id.unique()
    s_max = df_depth[df_depth.name == sname].depth_cm.max()
    
    gt_val = "N/A"
    if not df_annot.empty:
        sa = df_annot[df_annot.sensor_id.isin(dep_ids)]
        if not sa.empty:
            gt_val = f"{sa.annot_mid.max():.1f}"
            
    vlm_val = "N/A"
    if not df_vlm_adv.empty:
        sv = df_vlm_adv[(df_vlm_adv.sensor_id.isin(dep_ids)) & (df_vlm_adv.parsed_choice != 'N/A')]
        if not sv.empty:
            vlm_val = f"{sv.annot_mid.max():.1f}"
            
    stats.append({
        'Location': sname,
        'Sensor Peak [cm]': f"{s_max:.1f}",
        'GT Peak [cm]': gt_val,
        'VLM Peak [cm]': vlm_val
    })

df_stats = pd.DataFrame(stats).set_index('Location')
display(df_stats)

